In [1]:

# Imports

import helpers.helper_functions as hf
import mne
import os.path as op
from mne.channels import combine_channels
import pandas as pd
from mne.beamformer import make_lcmv, apply_lcmv_epochs
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import hilbert
import helpers.test_circ_plot as circ_plot
import gc
import helpers.stc_helper as stc_helper
import time
from pycircstat2.hypothesis import rayleigh_test
ss = hf.settings_dict()
from mne.stats import fdr_correction

In [2]:
stim_tmin = 0.7
stim_tmax = 3.7
base_tmin = -0.5
base_tmax = 0.0

In [3]:
for subject_index in ss['subject_idx_list']:

    # loop over each event type
    for event_id in ss['event_id_list']:

        event_name = str(event_id)
        duty_cycle = ss['event_name_list'][event_id - 1]
        subjects_dir = ss['fs_subjects_dir']
        subject = ss['subject_id_list'][subject_index]
        print("loading dataset for subject: ", subject)

        save_dir = Path(ss['hilbert_ref_dir']) / subject / event_name
        save_dir.mkdir(parents=True, exist_ok=True)

                # load stc data
        stim_stc_data = []
        base_stc_data = []
        for i in range(ss['n_trials']):
            hilbert_stc_file = Path(ss['hilbert_ref_dir']) / subject / event_name / f"{subject}-trial{i}-ret-ref-wrp-vol.stc"
            stc = mne.read_source_estimate(hilbert_stc_file)
            stim_stc = stc.copy().crop(tmin=stim_tmin, tmax=stim_tmax)
            base_stc = stc.copy().crop(tmin=base_tmin, tmax=base_tmax)
            stim_stc_data.append(stim_stc.data)
            base_stc_data.append(base_stc.data)

        stim_stc_data_stacked = np.stack(stim_stc_data)
        base_stc_data_stacked = np.stack(base_stc_data)

        # stc.data shape: (n_voxels, n_times) — phase differences in radians [-pi, pi]
        n_voxels, n_times = stc.data.shape

        z_stats = np.zeros(n_voxels)
        p_val = np.zeros(n_voxels)

        # Rayleigh test to get z-scores

        df = pd.read_csv(Path(ss['stc_dir']) / subject / event_name / f"{subject}-event-{event_name}-mask.csv")
        stc_mask = df["voxels"].values

        for voxel_idx in range(n_voxels):
            avg_stim = np.angle(np.exp(1j * stim_stc_data_stacked[:, voxel_idx]).mean(axis=1))
            avg_base = np.angle(np.exp(1j * base_stc_data_stacked[:, voxel_idx]).mean(axis=1))

            result_stim = rayleigh_test(avg_stim)    # angles in radians
            result_base = rayleigh_test(avg_base)    # angles in radians
            # apply mask before adding results to the list
            if stc_mask[voxel_idx]:
                z_stats[voxel_idx] = 0 - result_base.z
                p_val[voxel_idx] = 1
                continue
            z_stats[voxel_idx] = result_stim.z - result_base.z
            p_val[voxel_idx] = result_stim.pval

        H0, p_val_corr = fdr_correction(p_val, 0.05)

        z_stc       = stc.copy().crop(0.0, 0.0 + 1/ss['fs'])
        z_stc.data  = z_stats[:, np.newaxis]   # (n_voxels, 1)

        p_stc       = stc.copy().crop(0.0, 0.0 + 1/ss['fs'])
        p_stc.data  = p_val_corr[:, np.newaxis]   # (n_voxels, 1)

        z_stc.save(save_dir / f"{subject}-event-{event_name}-z-vol.stc" , overwrite=True)
        p_stc.save(save_dir / f"{subject}-event-{event_name}-p-vol.stc" , overwrite=True)

        print(f"z min  : {z_stats.min():.4f}")
        print(f"z max  : {z_stats.max():.4f}")
        print(f"z mean : {z_stats.mean():.4f}")
        print(f"Voxels > 0   : {(z_stats > 0).sum()} / {n_voxels}")

        print(f"p min  : {p_val_corr.min():.4f}")
        print(f"p max  : {p_val_corr.max():.4f}")
        print(f"p mean : {p_val_corr.mean():.4f}")
        print(f"Significtant voxels   : {(p_val_corr < 0.05).sum()} / {n_voxels}")


loading dataset for subject:  0005_3SJ
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
z min  : -4.7367
z max  : 8.8245
z mean : -0.0237
Voxels > 0   : 229 / 1440
p min  : 0.0006
p max  : 1.0000
p mean : 0.8786
Significtant voxels   : 136 / 1440
loading dataset for subject:  0005_3SJ
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
z min  : -6.8059
z max  : 9.6019
z mean : 0.3199
Voxels > 0   : 364 / 1440
p min  : 0.0002
p max  : 1.0000
p mean : 0.7843
Significtant voxels   : 222 / 1440
loading dataset for subject:  0005_3SJ
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
z min  : -5.6180
z max  : 9.2710
z mean : -0.1613
Voxels > 0  